# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [3]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [4]:
df.columns.to_list()


['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [5]:
df.shape

(541909, 8)

In [6]:
df.info

<bound method DataFrame.info of        InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1         536365     71053                  WHITE METAL LANTERN         6   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
...          ...       ...                                  ...       ...   
541904    581587     22613          PACK OF 20 SPACEBOY NAPKINS        12   
541905    581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
541906    581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
541907    581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
541908    581587     22138        BAKING SET 9 PIECE RETROSPOT          3   

               InvoiceDate  UnitPrice  Cust

## A.2. Missing values & Duplicate data

In [8]:

df.duplicated().sum()
df=df.drop_duplicates()

In [9]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64

## A.3. Invalid values

In [15]:
invalid_q = (df['Quantity'] <= 0).sum()
invalid_p = (df['UnitPrice'] <= 0).sum()


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [14]:

df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()

df_clean['Sales'] = df_clean['Quantity'] * df_clean['UnitPrice']
df.head(2)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [16]:
print("Mean Sales:", df_clean['Sales'].mean())
print("Median Sales:", df_clean['Sales'].median())
print("Mode Sales:", df_clean['Sales'].mode()[0])

Mean Sales: 20.275398862211794
Median Sales: 9.92
Mode Sales: 15.0


## Group 2 — Dispersion

In [17]:
print("Range:", df_clean['Sales'].max() - df_clean['Sales'].min())
print("IQR:", df_clean['Sales'].quantile(0.75) - df_clean['Sales'].quantile(0.25))
print("Variance:", df_clean['Sales'].var())
print("Standard Deviation:", df_clean['Sales'].std())


Range: 168469.59900000002
IQR: 13.800000000000002
Variance: 73817.39396839004
Standard Deviation: 271.69356629922254


## Group 3 — Location and Shape

In [18]:
print("Percentiles (25%, 50%, 75%):")
print(df_clean['Sales'].quantile([0.25, 0.50, 0.75]))
print("Skewness:", df_clean['Sales'].skew())
print("Kurtosis:", df_clean['Sales'].kurtosis())


Percentiles (25%, 50%, 75%):
0.25     3.90
0.50     9.92
0.75    17.70
Name: Sales, dtype: float64
Skewness: 504.2325621167678
Kurtosis: 294741.14297274407


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [19]:
df.head(2)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [21]:
country_sales = df_clean.groupby('Country')['Sales'].sum().reset_index()
country_sales['Percentage'] = (country_sales['Sales'] / country_sales['Sales'].sum()) * 100
country_sales = country_sales.sort_values('Sales', ascending=False)
print(country_sales.head(10))

           Country        Sales  Percentage
36  United Kingdom  9001744.094   84.586078
24     Netherlands   285446.340    2.682234
10            EIRE   283140.520    2.660567
14         Germany   228678.400    2.148807
13          France   209625.370    1.969772
0        Australia   138453.810    1.301000
31           Spain    61558.560    0.578443
33     Switzerland    57067.600    0.536243
3          Belgium    41196.340    0.387107
32          Sweden    38367.830    0.360528


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [22]:
top_products = df_clean.groupby(['StockCode', 'Description'])['Sales'].sum().reset_index()
top_products = top_products.sort_values('Sales', ascending=False)
print(top_products.head(10))


     StockCode                         Description      Sales
4150       DOT                      DOTCOM POSTAGE  206248.77
1340     22423            REGENCY CAKESTAND 3 TIER  174156.54
2668     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60
3640    85123A  WHITE HANGING HEART T-LIGHT HOLDER  104284.24
2877     47566                       PARTY BUNTING   99445.23
3619    85099B             JUMBO BAG RED RETROSPOT   94159.81
2123     23166      MEDIUM CERAMIC TOP STORAGE JAR   81700.92
4153      POST                             POSTAGE   78101.88
4151         M                              Manual   77750.27
2029     23084                  RABBIT NIGHT LIGHT   66870.03


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [24]:
df_clean["YearMonth"]= df_clean['InvoiceDate'].dt.to_period('M')
monthly_sales = df_clean.groupby('YearMonth')['Sales'].sum()
print(monthly_sales)

YearMonth
2010-12     821452.730
2011-01     689811.610
2011-02     522545.560
2011-03     716215.260
2011-04     536968.491
2011-05     769296.610
2011-06     760547.010
2011-07     718076.121
2011-08     757841.380
2011-09    1056435.192
2011-10    1151263.730
2011-11    1503866.780
2011-12     637790.330
Freq: M, Name: Sales, dtype: float64


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [25]:

aov_country = df_clean.groupby('Country').agg(
    total_sales=('Sales', 'sum'),
    total_orders=('InvoiceNo', 'nunique')
)
aov_country['AOV'] = aov_country['total_sales'] / aov_country['total_orders']
print(aov_country.sort_values('AOV', ascending=False).head(10))


             total_sales  total_orders          AOV
Country                                            
Singapore       21279.29             7  3039.898571
Netherlands    285446.34            94  3036.663191
Australia      138453.81            57  2429.014211
Japan           37416.37            19  1969.282632
Lebanon          1693.88             1  1693.880000
Hong Kong       15483.00            11  1407.545455
Brazil           1143.60             1  1143.600000
Sweden          38367.83            36  1065.773056
Switzerland     57067.60            54  1056.807407
Denmark         18955.34            18  1053.074444


## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [27]:
cancel_rates = df.assign(is_cancelled=df['Quantity'] < 0).pivot_table(
    index='Country', 
    values='is_cancelled', 
    aggfunc=['count', 'mean']
)


cancel_rates.columns = ['count', 'cancel_rate']

cancel_rates = cancel_rates[cancel_rates['count'] > 100].sort_values('cancel_rate', ascending=False)
print(cancel_rates.head(10))

           count  cancel_rate
Country                      
USA          291     0.384880
Malta        127     0.118110
Japan        358     0.103352
Australia   1258     0.058824
Italy        803     0.056040
Germany     9480     0.047785
EIRE        8184     0.036779
Poland       341     0.032258
Singapore    229     0.030568
Sweden       461     0.023861


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Thị trường United Kingdom áp đảo hoàn toàn khi đóng góp hơn 80% tổng doanh thu của hệ thống e-commerce. Doanh số có tính mùa vụ rõ rệt, tăng trưởng mạnh vào các tháng cuối năm (tháng 10, 11) do nhu cầu mua sắm lễ tết. Tuy nhiên, dữ liệu có độ lệch lớn (skewness cao) do xuất hiện nhiều đơn hàng bán buôn giá trị cực lớn. Tỷ lệ hoàn/hủy hàng có sự khác biệt giữa các thị trường quốc tế, đòi hỏi chiến lược quản lý vận chuyển phù hợp.